# FedBabu Client

> The core abstraction for `FedBabu` client: [https://openreview.net/forum?id=HuaYQfggn5u](https://openreview.net/forum?id=HuaYQfggn5u)

In [ ]:
#| default_exp clients.fedbabu

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import copy
import random
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from fedai.clients.base_client import BaseClient
from fedai.utils.registery import AlgorithmRegistry

<torch._C.Generator>

In [ ]:
#| export
@AlgorithmRegistry.register_client("fedbabu")
class ClientFedBabu(BaseClient):
    def __init__(self,
                 id, # Unique identifier for the client
                 cfg, # A configuration object containing all the necessary parameters for training and evaluation.
                 train_loader, 
                 test_loader,
                 state, # A dictionary containing the model, optimizer and any changing component needed.
                 criterion,
                 device= None,
                 t= 0,
                 **kwargs # extra client-specific parameters (that cannot be passed in state and cfg), can be passed as here.
                 ):  
                 
        super().__init__(id, cfg, train_loader, test_loader, state, criterion, device, t, **kwargs)

        for param in self.model.head.parameters():
            param.requires_grad = False
            

### Training

In [ ]:
#| export
@patch
def fit(self: ClientFedBabu):
    
    self.model.to(self.device)
    self.model.train()
   
    for _ in range(self.cfg.local_epochs):
        for i, batch in enumerate(self.train_loader):
            self.optimizer.zero_grad()

            batch = self.send_to_device(batch)
            X, y = batch[self.data_key], batch[self.label_key]
            logits = self.model(X)
            loss = self.criterion(logits, y)
            loss.backward()
            self.optimizer.step()


In [ ]:
#| export
@patch
def fine_tune(self: ClientFedBabu, which_module=['backbone', 'head']):    
    
    self.model.train()
    self.model.to(self.device)

    if 'head' in which_module:
        for param in self.model.head.parameters():
            param.requires_grad = True

    if 'backbone' not in which_module:
        for param in self.model.backbone.parameters():
            param.requires_grad = False

    for epoch in range(self.cfg.algorithm.fine_tuning_epochs):
        for batch in self.train_loader:
            batch = self.send_to_device(batch)
            X, y = batch[self.data_key], batch[self.label_key]
            output = self.model(X)
            loss = self.loss(output, y)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

### Saving State

In [ ]:
#| export
@patch
def save_state(self: ClientFedBabu, save_to_disk= False):  

    state_dict = {}
    state_dict['model'] = self.model.state_dict()
    state_dict['optimizer'] = self.optimizer.state_dict()

    if save_to_disk:
        state_path = os.path.join(self.cfg.server.save_dir, str(self.t), f"local_output_{self.id}", "state.pth")
        if not os.path.exists(os.path.dirname(state_path)):
            os.makedirs(os.path.dirname(state_path))

        torch.save(state_dict, state_path)

    return state_dict

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()